In [17]:
from google.colab import files
uploaded = files.upload()


Saving heartattack.csv to heartattack (1).csv
Saving anemia.csv to anemia (1).csv
Saving diabetese.csv to diabetese (1).csv


In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


In [19]:
heart = pd.read_csv("heartattack.csv")

X = heart.drop("Target", axis=1)
y = heart["Target"]

# Save feature structure
X_heart = X.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler_heart = StandardScaler()
X_train = scaler_heart.fit_transform(X_train)
X_test = scaler_heart.transform(X_test)

model_heart = RandomForestClassifier(n_estimators=300, random_state=42)
model_heart.fit(X_train, y_train)

print("Heart Model Accuracy:", accuracy_score(y_test, model_heart.predict(X_test)))


Heart Model Accuracy: 0.99


In [20]:
anemia = pd.read_csv("anemia.csv")

X = anemia.drop("Result", axis=1)
y = anemia["Result"]

model_anemia = RandomForestClassifier(n_estimators=200, random_state=42)
model_anemia.fit(X, y)

print("Anemia Model Training Completed")


Anemia Model Training Completed


In [21]:
diabetes = pd.read_csv("diabetese.csv")

# Convert Gender M/F → 1/0
if diabetes["Gender"].dtype == object:
    diabetes["Gender"] = diabetes["Gender"].map({'M':1, 'F':0})

X = diabetes.drop("Diagnosis", axis=1)
y = diabetes["Diagnosis"]

# Save feature structure
X_diabetes = X.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler_diabetes = StandardScaler()
X_train = scaler_diabetes.fit_transform(X_train)
X_test = scaler_diabetes.transform(X_test)

model_diabetes = RandomForestClassifier(n_estimators=300, random_state=42)
model_diabetes.fit(X_train, y_train)

print("Diabetes Model Accuracy:", accuracy_score(y_test, model_diabetes.predict(X_test)))


Diabetes Model Accuracy: 0.8024539877300614


In [22]:
heart_tests = pd.DataFrame([
    {"Age":35,"Sex":0,"ChestPain":1,"Restbps":120,"Chol":180,"Fbs":0,"RestECG":0,
     "MaxHR":170,"Exang":0,"Oldpeak":0.2,"Slope":1,"Ca":0,"Thal":0},

    {"Age":52,"Sex":1,"ChestPain":3,"Restbps":135,"Chol":240,"Fbs":0,"RestECG":1,
     "MaxHR":145,"Exang":0,"Oldpeak":1.2,"Slope":2,"Ca":1,"Thal":2},

    {"Age":60,"Sex":1,"ChestPain":4,"Restbps":160,"Chol":280,"Fbs":1,"RestECG":2,
     "MaxHR":120,"Exang":1,"Oldpeak":3.5,"Slope":3,"Ca":3,"Thal":3}
])

heart_tests = heart_tests.reindex(columns=X_heart.columns, fill_value=0)
heart_scaled = scaler_heart.transform(heart_tests)
heart_prob = model_heart.predict_proba(heart_scaled)

for i,p in enumerate(heart_prob):
    chance = p[1]*100
    print(f"\nHeart Patient {i+1} → Disease Chance: {chance:.2f}%")



Heart Patient 1 → Disease Chance: 35.67%

Heart Patient 2 → Disease Chance: 99.67%

Heart Patient 3 → Disease Chance: 100.00%


In [23]:
anemia_tests = [
    [0,13.8,29.5,33.5,90],
    [1,11.5,27.0,32.0,82],
    [0,8.9,24.5,29.0,70]
]

anemia_prob = model_anemia.predict_proba(anemia_tests)

for i,p in enumerate(anemia_prob):
    chance = p[1]*100
    print(f"\nAnemia Patient {i+1} → Anemia Chance: {chance:.2f}%")



Anemia Patient 1 → Anemia Chance: 0.00%

Anemia Patient 2 → Anemia Chance: 98.00%

Anemia Patient 3 → Anemia Chance: 90.00%


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [24]:
diabetes_tests = pd.DataFrame([
    {"Age":30,"Gender":0,"BMI":22.4,"Chol":170,"TG":90,"HDL":60,"LDL":90,"Cr":0.8,"BUN":14},
    {"Age":48,"Gender":1,"BMI":28.9,"Chol":220,"TG":150,"HDL":45,"LDL":140,"Cr":1.1,"BUN":22},
    {"Age":58,"Gender":1,"BMI":34.5,"Chol":280,"TG":210,"HDL":35,"LDL":190,"Cr":1.9,"BUN":32}
])

diabetes_tests = diabetes_tests.reindex(columns=X_diabetes.columns, fill_value=0)
diabetes_scaled = scaler_diabetes.transform(diabetes_tests)
diabetes_prob = model_diabetes.predict_proba(diabetes_scaled)

for i,p in enumerate(diabetes_prob):
    chance = p[1]*100
    print(f"\nDiabetes Patient {i+1} → Diabetes Risk Chance: {chance:.2f}%")



Diabetes Patient 1 → Diabetes Risk Chance: 63.67%

Diabetes Patient 2 → Diabetes Risk Chance: 55.00%

Diabetes Patient 3 → Diabetes Risk Chance: 79.00%


In [25]:
!pip install xgboost



In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Models
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier


In [27]:
heart = pd.read_csv("heartattack.csv")

X = heart.drop("Target", axis=1)
y = heart["Target"]

X_heart = X.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler_heart = StandardScaler()
X_train = scaler_heart.fit_transform(X_train)
X_test = scaler_heart.transform(X_test)

# ---- SVM Model ----
model_heart_svm = SVC(kernel='rbf', probability=True, random_state=42)
model_heart_svm.fit(X_train, y_train)

y_pred = model_heart_svm.predict(X_test)

print("Heart Disease SVM Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Heart Disease SVM Accuracy: 0.965
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         7
           1       0.96      1.00      0.98       193

    accuracy                           0.96       200
   macro avg       0.48      0.50      0.49       200
weighted avg       0.93      0.96      0.95       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [28]:
anemia = pd.read_csv("anemia.csv")

X = anemia.drop("Result", axis=1)
y = anemia["Result"]

# ---- Logistic Regression ----
model_anemia_lr = LogisticRegression(max_iter=1000)
model_anemia_lr.fit(X, y)

print("Anemia Logistic Regression Training Completed")


Anemia Logistic Regression Training Completed


In [29]:
diabetes = pd.read_csv("diabetese.csv")

# Convert Gender M/F → 1/0
if diabetes["Gender"].dtype == object:
    diabetes["Gender"] = diabetes["Gender"].map({'M':1, 'F':0})

X = diabetes.drop("Diagnosis", axis=1)
y = diabetes["Diagnosis"]

X_diabetes = X.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler_diabetes = StandardScaler()
X_train = scaler_diabetes.fit_transform(X_train)
X_test = scaler_diabetes.transform(X_test)

# ---- XGBoost Model ----
model_diabetes_xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

model_diabetes_xgb.fit(X_train, y_train)

y_pred = model_diabetes_xgb.predict(X_test)

print("Diabetes XGBoost Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Diabetes XGBoost Accuracy: 0.7865030674846626
              precision    recall  f1-score   support

           0       0.84      0.87      0.85       583
           1       0.64      0.58      0.61       232

    accuracy                           0.79       815
   macro avg       0.74      0.72      0.73       815
weighted avg       0.78      0.79      0.78       815



In [30]:
heart_test_new = pd.DataFrame([
    # Patient 1 — Very Low Risk
    {"Age":29,"Sex":0,"ChestPain":1,"Restbps":110,"Chol":160,"Fbs":0,"RestECG":0,
     "MaxHR":185,"Exang":0,"Oldpeak":0.0,"Slope":1,"Ca":0,"Thal":0},

    # Patient 2 — Borderline Risk
    {"Age":50,"Sex":1,"ChestPain":2,"Restbps":130,"Chol":210,"Fbs":0,"RestECG":1,
     "MaxHR":155,"Exang":0,"Oldpeak":0.8,"Slope":2,"Ca":0,"Thal":1},

    # Patient 3 — Critical Risk
    {"Age":67,"Sex":1,"ChestPain":4,"Restbps":175,"Chol":310,"Fbs":1,"RestECG":2,
     "MaxHR":105,"Exang":1,"Oldpeak":4.2,"Slope":3,"Ca":3,"Thal":3}
])

heart_test_new = heart_test_new.reindex(columns=X_heart.columns, fill_value=0)
heart_scaled_new = scaler_heart.transform(heart_test_new)
heart_prob_new = model_heart_svm.predict_proba(heart_scaled_new)   # or model_heart

for i,p in enumerate(heart_prob_new):
    print(f"\nHeart Patient {i+1} → Disease Chance: {p[1]*100:.2f}%")



Heart Patient 1 → Disease Chance: 51.79%

Heart Patient 2 → Disease Chance: 58.54%

Heart Patient 3 → Disease Chance: 100.00%


In [31]:
anemia_test_new = [
    [1, 14.5, 30.0, 34.0, 92],   # Healthy Male
    [0, 12.0, 28.0, 32.5, 85],   # Slightly Low
    [1, 9.2, 25.0, 30.0, 75]     # Strong Anemia
]

anemia_prob_new = model_anemia_lr.predict_proba(anemia_test_new)  # or model_anemia

for i,p in enumerate(anemia_prob_new):
    print(f"\nAnemia Patient {i+1} → Anemia Chance: {p[1]*100:.2f}%")



Anemia Patient 1 → Anemia Chance: 1.91%

Anemia Patient 2 → Anemia Chance: 37.10%

Anemia Patient 3 → Anemia Chance: 99.90%


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [32]:
diabetes_test_new = pd.DataFrame([
    # Patient 1 — Very Healthy
    {"Age":26,"Gender":0,"BMI":20.8,"Chol":165,"TG":80,"HDL":65,"LDL":85,"Cr":0.7,"BUN":12},

    # Patient 2 — Warning Stage
    {"Age":44,"Gender":1,"BMI":27.5,"Chol":205,"TG":140,"HDL":48,"LDL":135,"Cr":1.0,"BUN":20},

    # Patient 3 — High Risk
    {"Age":62,"Gender":1,"BMI":36.2,"Chol":295,"TG":230,"HDL":33,"LDL":185,"Cr":2.1,"BUN":34}
])

diabetes_test_new = diabetes_test_new.reindex(columns=X_diabetes.columns, fill_value=0)
diabetes_scaled_new = scaler_diabetes.transform(diabetes_test_new)
diabetes_prob_new = model_diabetes_xgb.predict_proba(diabetes_scaled_new)  # or model_diabetes

for i,p in enumerate(diabetes_prob_new):
    print(f"\nDiabetes Patient {i+1} → Risk Chance: {p[1]*100:.2f}%")



Diabetes Patient 1 → Risk Chance: 84.77%

Diabetes Patient 2 → Risk Chance: 76.57%

Diabetes Patient 3 → Risk Chance: 93.76%
